# MNIST demo with CREST 

In [1]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

#### Load MNIST Datasets

In [2]:
from crest import DataServer

# Load datasets from server
ds_train,ds_valid,ds_test,ds_pred = DataServer.load('mnist')

# Get input/output labels
[INP,OUT] = ds_train.labels

# Num classes
n_classes = 10
patch_size = 28

#### Create the HierarchalTensorGraph

In [3]:
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
import tensorflow as tf
from crest import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

# Define the tensorflow layers, and convert to HTG objects
layers = [crest_layer(layer) for layer in [
    Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
    Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
    Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
    Flatten(),                                        # 28x28x10 -> 7840
    Dropout(0.3),                                     # 30% of data dropped
    Dense(n_classes, activation='softmax'),           # Class probabilities
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})
    
# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='MNIST',
    inputs={INP : tf.TensorSpec(shape=[None,patch_size, patch_size])},
    outputs={OUT : tf.TensorSpec(shape=[None,10])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

[None, None, None, None, None, None, None]


#### Build a model using the HTG

In [4]:
from crest import Model

# Define a custom metric if desired
def custom_metric(y_true, y_pred):
    squared_difference = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_difference, axis=-1)

# Set build options
build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'sparse_categorical_crossentropy', 
    'metrics'   : ['sparse_categorical_accuracy', custom_metric]
}

# Create anad build a model using the HTG
model = Model(graph)
model.build(**build_kwargs)

[<function sparse_categorical_accuracy at 0x16991f880>, <function custom_metric at 0x103e23910>]


#### Train the model

In [5]:
# Create the training and validation batchers
from crest.data.Batcher import Batcher

batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)

with batch_train as train, batch_valid as valid:
    model.fit(train, **{
    'validation_data'  : valid,
    'steps_per_epoch'  : 250, # 50k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1
    })

Epoch 1/5
  1/250 [..............................] - ETA: 39s - loss: 67.8292 - sparse_categorical_accuracy: 0.0800 - custom_metric: 29.3374

2023-11-27 10:54:34.712472: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


250/250 [==============================] - 3s 11ms/step - loss: 3.4638 - sparse_categorical_accuracy: 0.8712 - custom_metric: 27.3593 - val_loss: 0.5245 - val_sparse_categorical_accuracy: 0.9610 - val_custom_metric: 27.4872
Epoch 2/5
250/250 [==============================] - 3s 11ms/step - loss: 0.5309 - sparse_categorical_accuracy: 0.9418 - custom_metric: 27.3777 - val_loss: 0.3400 - val_sparse_categorical_accuracy: 0.9660 - val_custom_metric: 27.4871
Epoch 3/5
250/250 [==============================] - 3s 11ms/step - loss: 0.2862 - sparse_categorical_accuracy: 0.9560 - custom_metric: 27.2761 - val_loss: 0.2364 - val_sparse_categorical_accuracy: 0.9760 - val_custom_metric: 27.4870
Epoch 4/5
250/250 [==============================] - 3s 12ms/step - loss: 0.1816 - sparse_categorical_accuracy: 0.9633 - custom_metric: 27.5354 - val_loss: 0.2232 - val_sparse_categorical_accuracy: 0.9770 - val_custom_metric: 27.4870
Epoch 5/5
250/250 [==============================] - 3s 11ms/step - loss: 

#### Evaluate the model

In [6]:
# Evaluate the model on the test set
batch_test = Batcher(ds_test, **batch_kwargs)
evaluate_kwargs= {
    'verbose' : 1,
    'return_dict' : True
}

with batch_test as batcher:
    res = model.evaluate(batcher, verbose=1, return_dict=1)

1/1 [==============================] - 0s 13ms/step - loss: 0.1028 - sparse_categorical_accuracy: 0.9800 - custom_metric: 26.3271


Accuracy on the full test set should be > ~97%.

### Make predictions

In [7]:
# Create batcher for predictions
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
    'shuffle'    : False
}) 

# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size' : 200,
    'verbose'    : False,
    'steps'      : 10000//200
}

# Generate predictions
with batch_pred as batcher:
    model_pred_batch = model.predict(batch_pred, **pred_kwargs) # Using a batcher
    model_pred_dataset = model.predict(ds_pred,**pred_kwargs)   # Using a dataset
    print('Difference:', (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Difference: 0.0


#### Adding additional output (e.g., lat and lon) when making a predictions

In [8]:
# Create inputs with additional features lat and lon
import numpy as np
from crest.data.loading import StructuredDataset
x = ds_pred.data[0]
lat = np.arange(len(x),dtype=float)
lon = np.arange(len(x),dtype=float)
x_aux = (lat,lon,x)
ds_aux = StructuredDataset(*x_aux, labels=['lat','lon',INP])

# Specify additional output as coords when making predictions
model.predict(ds_aux,coords=['lat','lon'],**pred_kwargs)

{'class': array([[4.5745730e-19, 1.1093718e-23, 2.6664885e-15, ..., 1.0000000e+00,
         9.0248352e-14, 6.8714802e-15],
        [2.5953005e-20, 3.0376218e-14, 1.0000000e+00, ..., 7.7663664e-27,
         4.6512455e-17, 1.5978508e-28],
        [6.1645933e-09, 9.9999440e-01, 8.1307988e-08, ..., 5.2202003e-09,
         3.4588106e-06, 6.8951306e-10],
        ...,
        [2.0872619e-18, 5.7904297e-14, 7.4565091e-20, ..., 2.6532151e-08,
         4.3031753e-07, 3.3318408e-08],
        [3.4452112e-17, 1.1822743e-16, 1.3160406e-19, ..., 2.0161555e-16,
         1.8232579e-06, 5.0266191e-16],
        [8.6681420e-19, 7.4986460e-27, 1.4368331e-16, ..., 4.1351240e-23,
         7.7810591e-16, 1.5531913e-22]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}

In [9]:
print('Elapsed time = ', time.time() - start_time, '[s]')

Elapsed time =  26.91889190673828 [s]
